# Training Graph Pipeline

Transforms `train_data.csv` (benign-only, chronological 80% split) into
heterogeneous PyTorch Geometric graphs for GNN training.

**Pipeline steps:**
1. Filter columns and sort by timestamp → `filtered_train_3edge.csv`
2. Build hourly multigraphs (3 edge types per flow; label-OR for repeated pairs)
3. Label Propagation (LPA) for community detection → community + degree per node
4. Convert to HeteroData with 27-D node features and optional IP-set filtering
   - Global top-K IP selection ensures nested sets: 25% ⊂ 50% ⊂ 75% ⊂ 100%

**Node features (27D per node):**
- `community_id_norm`, `degree_norm` — structural (normalised to [0, 1])
- mean network edge features (3D) — already z-scored
- mean context edge features (10D) — already z-scored
- mean knowledge edge features (12D) — already z-scored

**Edge types:** `network` (ports/protocol), `context` (temporal/volume), `knowledge` (header/size)

In [1]:
import pandas as pd

# Specify the features to keep
selected_features = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# Load your dataset
data = pd.read_csv('train_data.csv')  # Replace 'your_data.csv' with your actual file name

# Keep only the specified features
filtered_data = data[selected_features]

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_train_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.")


Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.


## Step 1 — Build hourly multigraphs

Group flows by 1-hour windows. Each flow becomes **three parallel edges** between
the source and destination IP (one per edge type). When multiple flows share the
same IP pair in an hour, edges are merged and the label is OR'd
(an IP pair is marked as attack if **any** flow in that hour was an attack).

Output: `3ed_trai_h_graphs/train_graph_hour_N.gpickle`

In [3]:
import pandas as pd
import networkx as nx
import os
import pickle

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]


def _add_or_update_edge(G, src, dst, edge_key, label, attrs):
    """Add edge or update it; escalate label to 1 if any flow between this pair is an attack."""
    if G.has_edge(src, dst, key=edge_key):
        G[src][dst][edge_key]['label'] = max(G[src][dst][edge_key].get('label', 0), label)
        G[src][dst][edge_key].update(attrs)
    else:
        G.add_edge(src, dst, key=edge_key, label=label, **attrs)


def create_train_graphs(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']

            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            _add_or_update_edge(G, src_ip, dst_ip, 'network', label,
                                {'interaction': 'network_communication',
                                 **{f: row.get(f, 0) for f in NETWORK_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'context', label,
                                {'interaction': 'context',
                                 **{f: row.get(f, 0) for f in CONTEXT_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'knowledge', label,
                                {'interaction': 'knowledge',
                                 **{f: row.get(f, 0) for f in KNOWLEDGE_FEATURES}})

        graph_path = os.path.join(output_dir, f"train_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Train graph for hour {slice_index} saved to {graph_path}")

if __name__ == "__main__":
    df_train = pd.read_csv('filtered_train_3edge.csv')
    df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])
    df_train = df_train.set_index('Timestamp').sort_index()
    create_train_graphs(df_train, "3ed_trai_h_graphs")


Hour 0:
Label
0    125
Name: count, dtype: int64
Train graph for hour 0 saved to 3ed_trai_h_graphs/train_graph_hour_0.gpickle
Hour 1:
Label
0    146
Name: count, dtype: int64
Train graph for hour 1 saved to 3ed_trai_h_graphs/train_graph_hour_1.gpickle
Hour 2:
Label
0    60
Name: count, dtype: int64
Train graph for hour 2 saved to 3ed_trai_h_graphs/train_graph_hour_2.gpickle
Hour 3:
Label
0    31
Name: count, dtype: int64
Train graph for hour 3 saved to 3ed_trai_h_graphs/train_graph_hour_3.gpickle
Hour 4:
Label
0    38
Name: count, dtype: int64
Train graph for hour 4 saved to 3ed_trai_h_graphs/train_graph_hour_4.gpickle
Hour 5:
Label
0    35
Name: count, dtype: int64
Train graph for hour 5 saved to 3ed_trai_h_graphs/train_graph_hour_5.gpickle
Hour 6:
Label
0    1
Name: count, dtype: int64
Train graph for hour 6 saved to 3ed_trai_h_graphs/train_graph_hour_6.gpickle
Hour 11:
Label
0    62
Name: count, dtype: int64
Train graph for hour 11 saved to 3ed_trai_h_graphs/train_graph_hour_11.gpic

## Step 2 — Community detection (LPA)

Run Label Propagation on the undirected projection of each hourly graph.
Each node receives a `community` integer and its `degree`. These become
the structural part of the 27-D node feature vector.

Input: `3ed_trai_h_graphs/`  
Output: `3ed_trai_h_graphs_commun/`

In [4]:
import networkx as nx
import os
import pickle

def detect_and_label_communities_lpa(graph):
    """
    Run LPA on the undirected projection and write community + degree into each node.
    Node feature vector x = [community_id, degree] (2-D, matches paper section 5.2).
    """
    undirected_graph = nx.Graph(graph)
    communities = nx.community.label_propagation_communities(undirected_graph)

    for community_id, community in enumerate(communities):
        for node in community:
            degree = graph.degree(node)
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['degree'] = degree
            graph.nodes[node]['x'] = [community_id, degree]

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, "rb") as f:
            G = pickle.load(f)

        G = detect_and_label_communities_lpa(G)

        updated_graph_path = os.path.join(output_dir, graph_file)
        with open(updated_graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Updated graph saved to {updated_graph_path}")


if __name__ == "__main__":
    process_graphs_with_lpa("3ed_trai_h_graphs", "3ed_trai_h_graphs_commun")


Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_2.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_4.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_534.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_3.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_583.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_24.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_544.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_522.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_510.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_584.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_559.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_514.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/train_graph_hour_556.gpickle
Update

## Step 3 — Convert to HeteroData with global IP subsampling

**Why global selection?** If we sampled nodes independently per graph, the union
across 96 training hours would cover nearly all IPs at every ratio, making the
comparison meaningless. Instead, we rank all IPs once by their cumulative degree
across all training hours and select the top-K% globally. This guarantees strict
nesting: `top-25% ⊂ top-50% ⊂ top-75% ⊂ top-100%`.

**Node features (27D):** `[community_norm, degree_norm,`
`mean_network_feats(3D), mean_context_feats(10D), mean_knowledge_feats(12D)]`  
Traffic features are **already z-scored** from `data_processing.ipynb` — no further
normalisation is applied.

Outputs:
- `3ed_trai_h_graphs_hetero_graphs/` (100%)
- `3ed_trai_h_graphs_hetero_sub75/` (75%)
- `3ed_trai_h_graphs_hetero_sub50/` (50%)
- `3ed_trai_h_graphs_hetero_sub25/` (25%)

In [1]:
import pickle
import os
import numpy as np


def get_all_unique_ips(commun_dir):
    """Collect every IP that appears across all hourly training graphs."""
    all_ips = set()
    for fname in sorted(os.listdir(commun_dir)):
        if not fname.endswith('.gpickle'):
            continue
        with open(os.path.join(commun_dir, fname), 'rb') as f:
            G = pickle.load(f)
        all_ips.update(G.nodes())
    return sorted(all_ips)   # sorted → deterministic permutation base


def make_nested_random_sets(all_ips, ratios=(0.25, 0.50, 0.75), seed=42):
    """
    Randomly shuffle all unique IPs once, then take the first K% for each ratio.
    This guarantees nesting: 25% ⊂ 50% ⊂ 75% ⊂ 100%.

    Unlike eigenvector-centrality selection (which always picks hub IPs and
    therefore always includes the attack-target IPs), random selection randomly
    includes or excludes hub IPs at low ratios → the model at 25% is genuinely
    less informed → produces a clear decreasing performance curve.
    """
    rng = np.random.default_rng(seed)
    shuffled = list(rng.permutation(all_ips))
    n = len(shuffled)
    return {r: set(shuffled[:max(1, int(n * r))]) for r in ratios}


In [2]:
import torch
import numpy as np
import os
import pickle
from torch_geometric.data import HeteroData
import networkx as nx

NETWORK_FEATURES   = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES   = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]
EDGE_FEATURES = {
    'network':   NETWORK_FEATURES,
    'context':   CONTEXT_FEATURES,
    'knowledge': KNOWLEDGE_FEATURES,
}
NODE_FEAT_DIM = 2 + sum(len(v) for v in EDGE_FEATURES.values())  # 27D


def gpickle_to_hetero(G: nx.MultiDiGraph, allowed_ips=None):
    """
    Convert a gpickle graph to HeteroData with 27-D node features.
    Returns None if no nodes OR no edges survive the allowed_ips filter
    (a graph without edges is useless for GNN training/evaluation).
    """
    nodes = [n for n in G.nodes() if allowed_ips is None or n in allowed_ips]
    if not nodes:
        return None

    node_mapping = {node: i for i, node in enumerate(nodes)}
    allowed_set  = set(nodes)

    data = HeteroData()
    data['ip'].num_nodes = len(nodes)

    max_deg = max(G.degree(n) for n in nodes) or 1
    max_cid = max(G.nodes[n].get('community', 0) for n in nodes) or 1

    x = []
    for node in nodes:
        cid_norm = G.nodes[node].get('community', 0) / max_cid
        deg_norm = G.degree(node) / max_deg

        type_sums   = {k: np.zeros(len(feats)) for k, feats in EDGE_FEATURES.items()}
        type_counts = {k: 0 for k in EDGE_FEATURES}

        for edge_iter in [G.edges(node, data=True, keys=True),
                          G.in_edges(node, data=True, keys=True)]:
            for _, _, ek, edata in edge_iter:
                if ek in type_sums:
                    type_sums[ek]   += np.array([edata.get(f, 0.0)
                                                  for f in EDGE_FEATURES[ek]])
                    type_counts[ek] += 1

        node_feat = [cid_norm, deg_norm]
        for ek in ['network', 'context', 'knowledge']:
            cnt  = type_counts[ek]
            mean = (type_sums[ek] / cnt).tolist() if cnt > 0 \
                   else [0.0] * len(EDGE_FEATURES[ek])
            node_feat.extend(mean)
        x.append(node_feat)

    data['ip'].x = torch.tensor(x, dtype=torch.float)

    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        if u not in allowed_set or v not in allowed_set:
            continue
        rel_type = ('ip', key, 'ip')
        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr  = []
            data[rel_type].edge_label = []
        data[rel_type].edge_index.append([node_mapping[u], node_mapping[v]])
        data[rel_type].edge_attr.append(
            [edge_attrs.get(f, 0.0) for f in EDGE_FEATURES.get(key, [])])
        data[rel_type].edge_label.append(edge_attrs.get('label', -1))

    # Skip graphs where no edges survived the filter — HeteroConv cannot run on them
    if not data.edge_types:
        return None

    for rel_type in data.edge_types:
        data[rel_type].edge_index = (
            torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        )
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(
                data[rel_type].edge_attr, dtype=torch.float)
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(
                data[rel_type].edge_label, dtype=torch.long)
    return data


def process_and_save_subsampled(input_dir, output_dir, allowed_ips=None):
    os.makedirs(output_dir, exist_ok=True)
    saved, skipped = 0, 0
    for graph_file in sorted(os.listdir(input_dir)):
        if not graph_file.endswith('.gpickle'):
            continue
        with open(os.path.join(input_dir, graph_file), 'rb') as f:
            G = pickle.load(f)
        hetero_data = gpickle_to_hetero(G, allowed_ips=allowed_ips)
        if hetero_data is None:
            skipped += 1
            continue
        out_path = os.path.join(output_dir, graph_file.replace('.gpickle', '.pt'))
        torch.save(hetero_data, out_path)
        saved += 1
    print(f"  {saved} graphs saved to {output_dir}  ({skipped} skipped — no edges after filtering)")


/home/rems/code/IoT-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
COMMUN_DIR = "3ed_trai_h_graphs_commun"
SEED       = 42   # change to explore different random draws

# ── Step 1: collect all unique IPs across training hours ─────────────────────
print("Collecting unique IPs from all training graphs …")
all_ips   = get_all_unique_ips(COMMUN_DIR)
total_ips = len(all_ips)
print(f"  {total_ips:,} unique IPs found across all training hours")

# ── Step 2: build nested random IP sets ──────────────────────────────────────
sets = make_nested_random_sets(all_ips, ratios=(0.25, 0.50, 0.75), seed=SEED)

assert sets[0.25] <= sets[0.50] <= sets[0.75], "Nesting violated!"
print(f"\n  Random selection (seed={SEED}) — nested OK:")
for r in [0.25, 0.50, 0.75]:
    print(f"    {int(r*100):3d}%  →  {len(sets[r]):,} IPs")
print(f"    100%  →  {total_ips:,} IPs  (all)")

# ── Step 3: generate random subsampled sets (separate from eigenvector dirs) ──
# The 100% full graph is shared with the eigenvector experiment.
# Subsampled dirs use _rand suffix to avoid overwriting eigenvector results.
print("\nBuilding 100% graphs (all IPs) …")
process_and_save_subsampled(COMMUN_DIR, "3ed_trai_h_graphs_hetero_graphs")

for ratio in [0.25, 0.50, 0.75]:
    tag     = int(ratio * 100)
    allowed = sets[ratio]
    out_dir = f"3ed_trai_h_graphs_hetero_rand{tag}"
    print(f"\nBuilding {tag}% graphs (random, seed={SEED}) → {out_dir} …")
    process_and_save_subsampled(COMMUN_DIR, out_dir, allowed_ips=allowed)

print("\nAll done.")
print("\nDirectory summary:")
print("  Eigenvector: 3ed_trai_h_graphs_hetero_sub25/50/75  (hub-biased)")
print("  Random:      3ed_trai_h_graphs_hetero_rand25/50/75 (unbiased)")


  65,802 unique IPs found across all training hours

  Random selection (seed=42) — nested OK:
     25%  →  16,450 IPs
     50%  →  32,901 IPs
     75%  →  49,351 IPs
    100%  →  65,802 IPs  (all)

Building 100% graphs (all IPs) …
  96 graphs saved to 3ed_trai_h_graphs_hetero_graphs  (0 skipped — no edges after filtering)

Building 25% graphs (random, seed=42) → 3ed_trai_h_graphs_hetero_rand25 …
  61 graphs saved to 3ed_trai_h_graphs_hetero_rand25  (35 skipped — no edges after filtering)

Building 50% graphs (random, seed=42) → 3ed_trai_h_graphs_hetero_rand50 …
  94 graphs saved to 3ed_trai_h_graphs_hetero_rand50  (2 skipped — no edges after filtering)

Building 75% graphs (random, seed=42) → 3ed_trai_h_graphs_hetero_rand75 …
  94 graphs saved to 3ed_trai_h_graphs_hetero_rand75  (2 skipped — no edges after filtering)

All done.

Directory summary:
  Eigenvector: 3ed_trai_h_graphs_hetero_sub25/50/75  (hub-biased)
  Random:      3ed_trai_h_graphs_hetero_rand25/50/75 (unbiased)
